# King County: Refactored Workflow

In [ ]:
#Import for the workflow

import pandas as pd
from sklearn.model_selection import train_test_split

from src.kc_cleaning import remove_implausible_rows
from src.kc_features import KingCountyFeatures
from src.kc_train import train_model, save_model, load_model

In [ ]:
# Load the data

raw = pd.read_csv("data/King_County_House_prices_dataset.csv")
print(raw.shape)

(21597, 21)


“Split before feature engineering, so the test set doesn’t influence the features.”


In [ ]:
#Spliting data in to train and test
#train_test_split shuffles the rows randomly and puts 30% into a test set. The remaining 70% go into a training set.

train_raw, test_raw = train_test_split(raw, test_size=0.3, random_state=42)

In [ ]:
#Builds the features from the training data and applies them to the training and test sets:
#Fit: learns the waterfront locations from the training data, after removing the 33-bedroom record.
#Transform: cleans both sets and adds center_distance and water_distance.
#Check: counts the missing values in the training features, which should be 0, and shows the first rows.

features = KingCountyFeatures().fit(remove_implausible_rows(train_raw))
X_train = features.transform(train_raw)
X_test = features.transform(test_raw)
print(X_train.isna().sum().sum(), "missing values in training features")
X_train.head()

0 missing values in training features


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_above,sqft_basement,zipcode,lat,long,sqft_living15,sqft_lot15,last_known_change,center_distance,water_distance
753,8682300890,8/28/2014,699800.0,2,2.50,2380,6600,1.0,0.0,0.0,...,2380,0,98053,47.7170,-122.020,1870,6600,2010,19.391272,9.343229
1418,8073000550,4/15/2015,1700000.0,4,3.75,3190,17186,2.0,1.0,4.0,...,3190,0,98178,47.5115,-122.246,2290,13496,1999,12.943085,0.000000
8178,7212680850,9/3/2014,258000.0,3,2.50,1730,6930,2.0,0.0,0.0,...,1730,0,98003,47.2621,-122.308,1780,6930,1994,41.002661,9.295898
2254,8880600070,11/12/2014,245000.0,4,2.00,1870,8750,1.0,0.0,2.0,...,1870,0,98022,47.1985,-122.001,1770,8207,1977,51.087178,29.521018
4063,7226500100,2/19/2015,373000.0,8,3.00,2850,12714,1.0,0.0,0.0,...,2850,0,98055,47.4859,-122.205,1480,4942,1959,16.030633,2.199755


In [ ]:
#Trains the model and keeps the best version:
#Train: runs the full pipeline on the training data, with cleaning, features, and ElasticNet, and tries 9 regularization settings using cross-validation.
#Report: prints the best settings found.


search = train_model(train_raw, train_raw["price"])
print("Best parameters:", search.best_params_)
model = search.best_estimator_

Best parameters: {'model__regressor__alpha': 0.01, 'model__regressor__l1_ratio': 0.2}


In [6]:
print("Test R^2:", round(model.score(test_raw, test_raw["price"]), 3))

Test R^2: 0.839


In [7]:
save_model(model, "model/model.bin")
loaded = load_model("model/model.bin")
print("Same predictions:", (loaded.predict(test_raw) == model.predict(test_raw)).all())

Same predictions: True
